# Ulaanbaatar Weather Data Exploration (1990-2024)

This notebook performs initial data exploration and preprocessing of Ulaanbaatar weather data.

## Objectives:
- Load and inspect raw weather data
- Check data quality
- Preprocess and clean data
- Create aggregated datasets (daily, monthly, yearly)
- Generate initial visualizations

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

# Add src to path
sys.path.append('../')

from config.config import *
from src.utils.data_preprocessing import *
from src.visualization.plots import *

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Plotting settings
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("Libraries loaded successfully!")

## 1. Load Raw Data

Load the Ulaanbaatar weather data from CSV file.

In [ ]:
# Load data
data_file = WEATHER_DATA_FILE

print(f"Loading data from: {data_file}")
df_raw = load_weather_data(data_file)

# Display first few rows
df_raw.head()

In [ ]:
# Display data info
print("\nDataset Information:")
df_raw.info()

In [ ]:
# Display summary statistics
print("\nSummary Statistics:")
df_raw.describe()

## 2. Data Preprocessing

Convert datetime, handle missing values, and add temporal features.

In [ ]:
# Run complete preprocessing pipeline
processed_data = preprocess_pipeline(data_file, save_processed=True)

# Extract datasets
df_hourly = processed_data['hourly']
df_daily = processed_data['daily']
df_monthly = processed_data['monthly']
df_yearly = processed_data['yearly']
quality_report = processed_data['quality_report']

In [ ]:
# Display processed hourly data
print("Processed Hourly Data:")
df_hourly.head()

In [ ]:
# Display daily aggregated data
print("Daily Aggregated Data:")
df_daily.head()

## 3. Initial Visualizations

In [ ]:
# Plot temperature trend (daily average)
plot_temperature_trend(
    df_daily, 
    column='temp_mean',
    title='Ulaanbaatar Daily Average Temperature (1990-2024)',
    save_path=FIGURES_DIR / '01_temperature_trend_daily.png'
)
plt.show()

In [ ]:
# Plot temperature distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df_daily['temp_mean'].dropna(), bins=50, 
             color=COLORS['primary'], alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Temperature (°C)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Temperature Distribution')
axes[0].grid(True, alpha=0.3)

# Box plot by month
df_hourly.boxplot(column='temp', by='month', ax=axes[1])
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Temperature (°C)')
axes[1].set_title('Temperature by Month')
plt.suptitle('')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_temperature_distribution.png', dpi=300)
plt.show()

In [ ]:
# Temperature statistics by decade
print("\nTemperature Statistics by Decade:")
decade_stats = df_hourly.groupby('decade_label')['temp'].agg([
    'count', 'mean', 'std', 'min', 'max'
]).round(2)
print(decade_stats)

In [ ]:
# Temperature statistics by season
print("\nTemperature Statistics by Season:")
season_stats = df_hourly.groupby('season')['temp'].agg([
    'count', 'mean', 'std', 'min', 'max'
]).round(2)
print(season_stats)

In [ ]:
# Plot temperature by decade
plot_temperature_by_decade(
    df_hourly,
    column='temp',
    save_path=FIGURES_DIR / '03_temperature_by_decade.png'
)
plt.show()

## 4. Yearly Temperature Summary

In [ ]:
# Display yearly data
print("Yearly Temperature Summary:")
print(df_yearly[['year', 'temp_mean', 'temp_min', 'temp_max']])

In [ ]:
# Plot yearly temperature
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(df_yearly['year'], df_yearly['temp_mean'], 
        marker='o', linewidth=2, markersize=6, 
        color=COLORS['primary'], label='Mean')
ax.fill_between(df_yearly['year'], 
                df_yearly['temp_min'], 
                df_yearly['temp_max'],
                alpha=0.2, color=COLORS['primary'],
                label='Min-Max Range')

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Temperature (°C)', fontsize=12)
ax.set_title('Yearly Temperature Summary (1990-2024)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '04_yearly_temperature.png', dpi=300)
plt.show()

## 5. Other Weather Variables

In [ ]:
# Plot other variables over time
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

variables = [
    ('humidity', 'Humidity (%)', axes[0, 0]),
    ('pressure', 'Pressure (hPa)', axes[0, 1]),
    ('wind_speed', 'Wind Speed (m/s)', axes[1, 0]),
    ('clouds_all', 'Cloud Coverage (%)', axes[1, 1])
]

for var, ylabel, ax in variables:
    if var in df_daily.columns:
        col = f'{var}_mean' if f'{var}_mean' in df_daily.columns else var
        ax.plot(df_daily.index, df_daily[col], alpha=0.5, linewidth=0.5)
        
        # Add moving average
        ma = df_daily[col].rolling(window=365, center=True).mean()
        ax.plot(df_daily.index, ma, color='red', linewidth=2, label='365-day MA')
        
        ax.set_ylabel(ylabel)
        ax.set_title(f'{ylabel} Over Time')
        ax.legend()
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '05_other_variables.png', dpi=300)
plt.show()

## Summary

This notebook completed:
- ✅ Data loading and inspection
- ✅ Data quality assessment
- ✅ Data preprocessing and aggregation
- ✅ Initial visualizations
- ✅ Statistical summaries by decade and season

**Next Steps:**
- Proceed to `02_temporal_analysis.ipynb` for trend and cyclical analysis
- Then to `03_correlation_analysis.ipynb` for external factors analysis